In [23]:
import logging
from pathlib import Path
from zipfile import ZipFile

from tqdm.notebook import tqdm

In [6]:
data_dir = Path(r'C:\projects\foursquare-data-loading\data')
out_dir = data_dir / r'processed\foursquare_data_packs'
zip_dir = out_dir.parent / f'{out_dir.stem}_archive'

delivery_year = 2025
delivery_month = 1

# create partitioned path to where data will be stored
out_dir = out_dir / f'delivery_year={delivery_year}' / f'delivery_month={delivery_month}'
zip_dir = zip_dir / f'delivery_year={delivery_year}' / f'delivery_month={delivery_month}'

out_dir, zip_dir

(WindowsPath('C:/projects/foursquare-data-loading/data/processed/foursquare_data_packs/delivery_year=2025/delivery_month=1'),
 WindowsPath('C:/projects/foursquare-data-loading/data/processed/foursquare_data_packs_archive/delivery_year=2025/delivery_month=1'))

In [26]:
# get the list of file geodatabases to archive
fgdb_lst = list(out_dir.rglob('*.gdb'))

# iterate the source file geodatabases
for fgdb_pth in tqdm(fgdb_lst):

    # create a path to save the zipped archive
    zip_pth = zip_dir / f'{fgdb_pth.parent.stem}.zip'

    # ensure the location to save the archive exists
    if not zip_pth.parent.exists():
        zip_pth.parent.mkdir(parents=True)
    
    logging.info(f'Starting to create an archive at {str(zip_pth)}')
    
    # build the archive
    with ZipFile(zip_pth, mode='w', compresslevel=9) as zipper:
    
        # iterate the files in the file geodatabase
        for gdb_file in fgdb_pth.rglob('*'):

            # ignore lock files...they create problems
            if not gdb_file.suffix == '.lock':
    
                # create a path in the archive with the file geodatabase
                target_pth = gdb_file.relative_to(fgdb_pth.parent)
        
                # add the file to the archive
                zipper.write(gdb_file, target_pth)
    
    logging.info(f'Successfully created archive.')

  0%|          | 0/242 [00:00<?, ?it/s]